# LoRA Adapter 合并

这个 notebook 说明为什么要合并 LoRA adapter，以及如何把 adapter 合并到基座模型。

LoRA adapter 只保存增量参数，不是完整模型。很多部署场景更喜欢完整 Hugging Face 模型目录，因此需要 merge。

## 1. 输入和输出

输入：

- `base_model_name_or_path`：基座模型。
- `adapter_path`：LoRA/QLoRA 训练产物。
- `merged_output_path`：合并后模型保存目录。

输出：

- 合并后的 Hugging Face 模型目录。
- tokenizer 文件。

合并后可以继续导出 GGUF、做 AWQ/GPTQ，或用 Transformers/vLLM 加载。

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

base_model_name_or_path = "Qwen/Qwen2.5-7B-Instruct"
adapter_path = "../../outputs/qwen2_5_7b/lora"
merged_output_path = "../../outputs/qwen2_5_7b/merged"

# 会加载大模型，确认环境后再执行。
# tokenizer = AutoTokenizer.from_pretrained(base_model_name_or_path, trust_remote_code=True)
# base_model = AutoModelForCausalLM.from_pretrained(
#     base_model_name_or_path,
#     torch_dtype=torch.bfloat16,
#     device_map="auto",
#     trust_remote_code=True,
# )
# model = PeftModel.from_pretrained(base_model, adapter_path)
# merged_model = model.merge_and_unload()
# merged_model.save_pretrained(merged_output_path, safe_serialization=True)
# tokenizer.save_pretrained(merged_output_path)

## 2. 常见错误

- adapter 和基座模型不匹配：训练时用的 base model 必须和合并时一致。
- 误以为 adapter 是完整模型：adapter 不能脱离基座模型单独推理。
- 合并后忘记保存 tokenizer：部署时 tokenizer 缺失会导致加载失败或模板错误。
- QLoRA 训练后直接导出 GGUF：通常要先 merge，再导出。